In [ ]:
# Data: ../csv/Fig2_C_Example_replayprobalit/  (the original loading code is kept below each data line, as comments)
import os, glob, pickle, re, sys   # standard imports
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'analysis_scripts'))   # the criticality library
import pandas as pd
import numpy as np
CSV = Path('..') / 'csv' / 'Fig2_C_Example_replayprobalit'
OUT = Path('..') / 'figure_output'
OUT.mkdir(exist_ok=True)


def read_csv(name, **kw):
    """A table of this notebook (round_trip: the floats exactly as written)."""
    return pd.read_csv(CSV / name, float_precision='round_trip', **kw)

In [ ]:
import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
from skimage import measure
import os
import pandas as pd
import scipy.io as sio
import logging

logging.getLogger().setLevel(logging.ERROR)
basepath =  r"W:\data\PPP\PPP8\day8"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="mouse"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
# ---- data (../csv) ----
save_dir = str(OUT)
nrem_epochs = nel.EpochArray(read_csv('states_NREM.csv')[['start', 'stop']].to_numpy())
awake_epochs = nel.EpochArray(read_csv('states_WAKE.csv')[['start', 'stop']].to_numpy())
rem_epochs = nel.EpochArray(read_csv('states_REM.csv')[['start', 'stop']].to_numpy())
epoch_df = read_csv('epochs.csv')
sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
spike_times = None
# ---- the same from the recordings, with analysis_scripts/: ----
# save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\complex_system\manifold"
# if not os.path.exists(save_dir):
#     os.makedirs(save_dir)
# # Create the folder if it doesn't exist
# if not os.path.exists(new_folder_path):
#     os.makedirs(new_folder_path)
#     print(f"Folder created: {new_folder_path}")
# else:
#     print(f"Folder already exists: {new_folder_path}")
# # load in spike data from hpc pyramidal cells
# st, cm = loading.load_spikes(
#     basepath, brainRegion=brain_region                          # putativeCellType="Pyr"
# )
# # load theta epochs
# state_dict = loading.load_SleepState_states(basepath)
# nrem_epochs = nel.EpochArray(
#     state_dict["NREMstate"],
# )
# awake_epochs = nel.EpochArray(
#     state_dict["WAKEstate"],
# )
# # theta_epochs = nel.EpochArray(
# #     state_dict["THETA"],
# # )
# rem_epochs=nel.EpochArray(state_dict["REMstate"],)
# try:
#     # Try default loader
#     epoch_df = loading.load_epoch(basepath)
# except Exception as e:
#     print(f"load_epoch failed with error: {e}")
#     print("Falling back to manual .mat file load...")
# 
#     # Load the .mat file
#     mat = sio.loadmat(os.path.join(basepath, "DayNight25.MergePoints.events.mat"), squeeze_me=True)
# 
#     # Extract the 'MergePoints' structure
#     merge_points = mat['MergePoints']
# 
#     # Extract the timestamps array: shape (N, 2)
#     timestamps = merge_points['timestamps']
#     # Unwrap the array inside
#     timestamps = np.array(timestamps.tolist())
# 
#     # shape (N, 2)
#     epoch_df = pd.DataFrame(timestamps, columns=["startTime", "stopTime"])
# 
# sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
# E_spikes = st.time[(cm.putativeCellType == 'Pyramidal Cell')]
# I_spikes= st.time[(cm.putativeCellType.str.contains("Interneuron"))]
# spike_times=st.time
# n_cell=len(spike_times)
# print(f"number of cells: {n_cell}")

In [ ]:
epoch_df

In [ ]:
import os, re, numpy as np, pandas as pd
import glob
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False,
            simplify_cells=True
        )
    except TypeError:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)
def _to_1d_array(x):
    """
    Robustly coerce MATLAB-loaded field to a 1D numpy array.
    - Scalars -> 1-length array
    - Lists/tuples/ndarrays -> flattened 1D
    - None / missing -> empty array
    """
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)  # start as object to avoid surprises
    if arr.ndim == 0:
        # scalar or unsized object
        return np.atleast_1d(arr).astype(float, copy=False)
    # flatten and try float cast (ignore non-numeric gracefully)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        # fallback: best-effort, convert elementwise
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    # MATLAB char arrays sometimes come in as numpy arrays
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        # arrays or weird containers -> join characters
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        # e.g., array of characters
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def load_d2_timeline_complete(basepath, region):
    """Loads all sleep-state data, including d2_err."""
    region = region.upper()
    search_dir = os.path.join(basepath, "complex_system")
    # pats = [f"*_{region}_D2_time_resolved_rippleablate.mat", f"*_{region}_D2_time_rippleablate.mat"]
    pats = [f"*_{region}_D2_time_resolved.mat", f"*_{region}_D2_time.mat"]
    files = []
    for pat in pats:
        files.extend(glob.glob(os.path.join(search_dir, pat)))
    if not files:
        raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")

    mat_path = max(files, key=os.path.getmtime)

    mat = _load_mat_any(mat_path)
    results = mat['results']
    params = mat.get('params', {})

    # Make results iterable no matter if it's a list, ndarray, or dict
    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))


    # Try to infer animal/day from filename if available
    fname = os.path.basename(mat_path)
    animal_name = None
    day = None
    m_day = re.search(r'day(\d+)', fname, flags=re.I)
    if m_day:
        day = int(m_day.group(1))
    m_animal = re.match(r'([^_]+)_day\d+_', fname)
    if m_animal:
        animal_name = m_animal.group(1)

    all_centers, all_d2, all_d2err, all_states = [], [], [], []
    n_states_with_err = 0

    for i, result_item in enumerate(iter_results):
        # Access as dict-like regardless of MATLAB struct quirks
        state = _to_state_str(result_item.get('state', 'UNKNOWN')) if hasattr(result_item, 'get') else \
                _to_state_str(result_item['state']) if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) else 'UNKNOWN'

        centers_raw = result_item.get('center', []) if hasattr(result_item, 'get') else \
                      (result_item['center'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'center' in result_item else [])
        d2_raw = result_item.get('d2', []) if hasattr(result_item, 'get') else \
                 (result_item['d2'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2' in result_item else [])
        d2err_raw = result_item.get('d2_err', []) if hasattr(result_item, 'get') else \
                    (result_item['d2_err'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2_err' in result_item else [])

        centers  = _to_1d_array(centers_raw)
        d2_vals  = _to_1d_array(d2_raw)
        d2_err   = _to_1d_array(d2err_raw)

        # Presence flag (some states may not have d2_err)
        has_err = d2_err.size > 0 and np.any(np.isfinite(d2_err))
        n_states_with_err += int(has_err)

        # Harmonize lengths: truncate all to the common min
        sizes = [centers.size, d2_vals.size] + ([d2_err.size] if has_err else [])
        n_pts = min(sizes) if min(sizes) > 0 else 0

        if n_pts == 0:
            continue

        centers = centers[:n_pts]
        d2_vals = d2_vals[:n_pts]
        d2_err  = (d2_err[:n_pts] if has_err else np.full(n_pts, np.nan, dtype=float))

        # Ranges
        try:
            pass
        except ValueError:
            pass

        try:
            pass
        except ValueError:
            pass

        if np.any(np.isfinite(d2_err)):
            try:
                dmin, dmax = float(np.nanmin(d2_err)), float(np.nanmax(d2_err))
            except ValueError:
                pass
        else:
            pass

        all_centers.append(centers)
        all_d2.append(d2_vals)
        all_d2err.append(d2_err)
        all_states.append(np.array([state]*n_pts, dtype=object))

    if len(all_centers) == 0:
        raise ValueError("No data found in any sleep state - all states are empty")

    # Concatenate
    all_centers = np.concatenate(all_centers)
    all_d2      = np.concatenate(all_d2)
    all_d2err   = np.concatenate(all_d2err)
    all_states  = np.concatenate(all_states)

    # DataFrame (include d2_err)
    df = pd.DataFrame({
        'center_s': all_centers.astype(float, copy=False),
        'd2':       all_d2.astype(float, copy=False),
        'd2_err':   all_d2err.astype(float, copy=False),
        'state':    all_states.astype(str)
    }).sort_values('center_s').reset_index(drop=True)


    # Empty-state tracker
    empty_states = []
    for ri in iter_results:
        st = _to_state_str(ri.get('state','UNKNOWN')) if hasattr(ri,'get') else \
             _to_state_str(ri['state']) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'state' in ri else 'UNKNOWN'
        csz = np.size(np.atleast_1d(ri.get('center', []))) if hasattr(ri,'get') else \
              np.size(np.atleast_1d(ri['center'])) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'center' in ri else 0
        if csz == 0:
            empty_states.append(st)

    meta = {
        "animal_name": animal_name,
        "day": day,
        "region": region,
        "path": mat_path,
        "params": params,
        "columns": list(df.columns),
        "shape": df.shape,
        "n_states": len(iter_results),
        "states_with_d2_err": n_states_with_err,
        "empty_states": empty_states
    }

    return df, meta



# ---- data (../csv) ----
ca1_complete_df = read_csv('d2_timeline_CA1.csv')
ca1_complete_meta = {}
# ---- the same from the recordings, with analysis_scripts/: ----
# try:
#     ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
#     
#     # pfc_complete_df, pfc_complete_meta = load_d2_timeline_complete(basepath, "PFC")
# 
#     
# except Exception as e:
#     print(f"❌ Failed: {e}")
#     import traceback
#     traceback.print_exc()

In [ ]:
epoch_df

In [ ]:
sleep_epochs

In [ ]:
ca1_complete_df

In [ ]:
# sleep_epochs = epoch_df.loc[
#     epoch_df["name"].str.contains("midsleep", case=False),
#     ["startTime", "stopTime"]
# ].values
# import numpy as np

# t = ca1_complete_df["center_s"].values

# mask = np.zeros(len(t), dtype=bool)
# for start, stop in sleep_epochs:
#     mask |= (t >= start) & (t <= stop)

# ca1_complete_df = ca1_complete_df.loc[mask].reset_index(drop=True)

In [ ]:
ca1_complete_df

In [ ]:
import numpy as np
from scipy.signal import welch
import nelpy as nel

def _flatten_spike_times(spike_times):
    """Accepts list/tuple/array of arrays or a single 1D array, returns sorted 1D float array."""
    if isinstance(spike_times, (list, tuple)):
        # list of arrays or scalars
        if len(spike_times) == 0:
            return np.array([], dtype=float)
        if hasattr(spike_times[0], "__len__") and not np.isscalar(spike_times[0]):
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in spike_times if st is not None and len(st) > 0])
        else:
            times = np.asarray(spike_times, dtype=float).ravel()
    else:
        arr = np.asarray(spike_times, dtype=object)
        if arr.dtype == object:
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in arr if st is not None and len(st) > 0])
        else:
            times = arr.astype(float).ravel()

    times = times[np.isfinite(times)]
    if times.size == 0:
        raise ValueError("No valid spike times found after flattening.")
    return np.sort(times)

def bin_spikes(spike_times, bin_size=0.04, t_start=None, t_stop=None):
    st = _flatten_spike_times(spike_times)
    if t_start is None: t_start = float(np.floor(st.min()))
    if t_stop  is None: t_stop  = float(np.ceil(st.max()))
    edges = np.arange(t_start, t_stop + bin_size, bin_size)
    counts, _ = np.histogram(st, bins=edges)
    rates = counts / bin_size
    t = edges[:-1] + bin_size/2
    return t, rates

# def segments_from_epochs(t, x, epochs):
#     arr = epochs.data if isinstance(epochs, nel.EpochArray) else np.asarray(epochs)
#     return [x[(t >= s) & (t < e)] for s, e in arr if np.any((t >= s) & (t < e))]

In [ ]:
# Create plot for random 200 second segment containing all states
import matplotlib.pyplot as plt
import numpy as np
import random

# First compute MUA from spike times

# Compute MUA firing rate with 0.04 second bins
# ---- data (../csv) ----
_mua = read_csv('mua_40ms.csv')
time_bins, firing_rate = _mua['time_bins'].to_numpy(), _mua['firing_rate'].to_numpy()
# ---- the same from the recordings, with analysis_scripts/: ----
# time_bins, firing_rate = bin_spikes(spike_times, bin_size=0.04)

In [ ]:
import numpy as np
import pandas as pd

# --- Inputs assumed to exist ---
# time_bins, firing_rate  # from bin_spikes(..., bin_size=0.04)
# ca1_complete_df          # has a 'center_s' column

# 1) Make sure we have bin *centers* that match firing_rate length
import numpy as np
import pandas as pd

# --- Inputs assumed to exist ---
# time_bins, firing_rate  # from bin_spikes(..., bin_size=0.04)
# ca1_complete_df         # has a 'center_s' column

# 1) Make sure we have bin *centers* that match firing_rate length
tb = np.asarray(time_bins, float).ravel()
fr = np.asarray(firing_rate, float).ravel()

if tb.size == fr.size + 1:            # looks like edges -> make centers
    t = 0.5 * (tb[:-1] + tb[1:])
elif tb.size == fr.size:               # already centers
    t = tb
else:
    raise ValueError("time_bins length must equal firing_rate length or be exactly one longer (edges).")

# 2) Clean + sort
ok = np.isfinite(t) & np.isfinite(fr)
t, fr = t[ok], fr[ok]
ordr = np.argsort(t)
t, fr = t[ordr], fr[ordr]

# 3) 30 s centered moving average AND standard deviation of MUA
dt = float(np.median(np.diff(t)))
win_s = 30.0
win_n = max(int(round(win_s / dt)), 1)

# Use pandas rolling for a centered window
fr_series = pd.Series(fr, index=pd.Index(t, name="t"))
roller = fr_series.rolling(window=win_n, center=True, min_periods=max(1, win_n//4))

fr_mean_30s = roller.mean().to_numpy()
fr_std_30s = roller.std().to_numpy()  # Calculate Std Dev on the same window

# 4) Interpolate both metrics onto the D² centers
centers = np.asarray(ca1_complete_df["center_s"], float)

mua_mean_at_centers = np.interp(centers, t, fr_mean_30s, left=np.nan, right=np.nan)
mua_std_at_centers = np.interp(centers, t, fr_std_30s, left=np.nan, right=np.nan)

# 5) Save to dataframe
ca1_complete_df["MUA"] = mua_mean_at_centers
ca1_complete_df["MUA_std"] = mua_std_at_centers

In [ ]:
ca1_complete_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Inputs assumed to exist ---
# E_spikes: array/list of excitatory spike times
# I_spikes: array/list of inhibitory spike times
# ca1_complete_df: has a 'center_s' column

import numpy as np
import pandas as pd

# --- Inputs assumed to exist ---
# E_spikes: array of Excitatory spike times
# I_spikes: array of Inhibitory spike times
# ca1_complete_df: has a 'center_s' column
# bin_spikes(spikes, bin_size, t_start=None, t_end=None): defined in the notebook



# 2. Bin spikes for both populations
# Note: passing t_start/t_end ensures alignment
# ---- data (../csv) ----
_E = read_csv('rate_E_40ms.csv')
time_bins, fr_E = _E['time_bins'].to_numpy(), _E['fr_E'].to_numpy()
fr_I = read_csv('rate_I_40ms.csv')['fr_I'].to_numpy()
# ---- the same from the recordings, with analysis_scripts/: ----
# time_bins, fr_E = bin_spikes(E_spikes, bin_size=0.04)
# _,         fr_I = bin_spikes(I_spikes, bin_size=0.04)

# 3. Clean and prepare for smoothing
# Use the centers of the bins
if len(time_bins) == len(fr_E) + 1:
    t_centers = 0.5 * (time_bins[:-1] + time_bins[1:])
else:
    t_centers = time_bins

# Ensure matching lengths (in case bin_spikes returns slight mismatches due to floating point)
n_bins = min(len(fr_E), len(fr_I), len(t_centers))
t_centers = t_centers[:n_bins]
fr_E = fr_E[:n_bins]
fr_I = fr_I[:n_bins]

# 4. Smooth E and I separately (30s window)
# Smoothing *before* division is critical to avoid instability from low firing rates
dt = 0.04
win_s = 30.0
win_n = max(int(round(win_s / dt)), 1)

temp_df = pd.DataFrame({
    'fr_E': fr_E,
    'fr_I': fr_I
}, index=t_centers)

# Centered rolling mean
temp_smooth = temp_df.rolling(
    window=win_n,
    center=True,
    min_periods=max(1, win_n//4)
).mean()

# 5. Compute E/I Ratio
# Add epsilon to prevent division by zero
epsilon = 1e-6
temp_smooth['EI_ratio'] = temp_smooth['fr_E'] / (temp_smooth['fr_I'] + epsilon)

# 6. Interpolate onto ca1_complete_df centers
centers = np.asarray(ca1_complete_df["center_s"], float)
t_smooth = temp_smooth.index.values
ei_smooth = temp_smooth['EI_ratio'].to_numpy()

ei_at_centers = np.interp(centers, t_smooth, ei_smooth, left=np.nan, right=np.nan)

# 7. Save to DataFrame
ca1_complete_df["EI_ratio"] = ei_at_centers

SWR and barrage

In [ ]:
# ---- data (../csv) ----
ripple_peaks = read_csv('ripple_peaks.csv')['ripple_peaks']
barrage_epochs = []
# ---- the same from the recordings, with analysis_scripts/: ----
# import numpy as np
# import scipy.io as sio      
# import scipy
# # Load ripple events
# ripple_events = loading.load_ripples_events(basepath=basepath)
# ripple_epochs = nel.EpochArray(np.array([ripple_events.start, ripple_events.stop]).T)
# ripple_peaks=ripple_events.peaks
#     # Load barrage events
# ripple_epochs=nrem_epochs&ripple_epochs
# try:
#     # Load barrage events
#     #brpath=r"W:\data\PPP\CA2\PPP7\day8"
#     barrage_events = loading.load_barrage_events(basepath=basepath)  
#     # Create barrage epochs
#     barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
#     # Find barrages in NREM epochs
#     barrages_in_nrem = barrage_epochs & nrem_epochs
#     mat_path = os.path.join(basepath, f'{basename}.HSEn2.events.mat')
# 
#     data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
#     barrage = data['HSEn2']
#     barrage_timestamps= barrage.timestamps 
#     barrage_epochs=nrem_epochs&barrage_epochs
# except Exception as e:
#     # General exception for unexpected errors
#     print(f"No barrage file: {e}")
#     barrage_epochs=[]
#     # Extract start times

In [ ]:
epoch_df

In [ ]:
# ---- data (../csv) ----
import numpy as np
import matplotlib.pyplot as plt
left_times = read_csv('replay_times_left.csv')['left_times'].tolist()
right_times = read_csv('replay_times_right.csv')['right_times'].tolist()
all_intervals = [np.array(iv) for iv in read_csv('replay_intervals.csv')[['start', 'stop']].to_numpy()]
# ---- the same from the recordings, with analysis_scripts/: ----
# import scipy.io
# import numpy as np
# from scipy.ndimage import zoom
# from scipy.interpolate import interp1d
# import umap
# import matplotlib.pyplot as plt
# 
# ## Load data
# mat_path = f'D:/Documents/GitHub/R_Project/PPP/codebook/Criticality_analysis/replay_results/{animal_name}_{day}_replay_sequences_new.mat'
# data = scipy.io.loadmat(mat_path)
# 
# # Define epochs and templates
# epochs = ['epoch1','epoch3','epoch5']  # only use epoch1,3,4
# template = 'template02'  # only use template04
# bin_size = 0.01  # 10 ms
# target_bins = 30  # fixed number of bins for all events
# 
# # Assume these are alre
# # ady loaded:
# # E_spikes: list of spike times (one per unit)
# # E_CellID: list of unit IDs corresponding to E_spikes
# 
# binned_firing_rates = []
# avg_pos = []
# epoch_labels = []
# left_replay = []
# right_replay = []
# left_replay = []
# left_times  = []
# 
# right_replay = []
# right_times  = []
# left_intervals=[]
# right_intervals=[]
# left_replay_goodfit=[]
# right_replay_goodfit=[]
# left_replay_slope=[]
# right_replay_slope=[]
# left_all_bestseq = []
# right_all_bestseq = []
# left_replay_matrix=[]
# right_replay_matrix=[]
# all_intervals=[]
# for e in epochs:
#     key_event = f'{e}_{template}_eventtimes'
#     key_seq   = f'{e}_{template}'
#     key_pval  = f'{e}_{template}_pvalue'
#     key_Goodfit  = f'{e}_{template}_Goodfit'
#     key_pval_r= f'{e}_{template}_rvalues'
#     key_slope  = f'{e}_{template}_slope'
#     key_best_seq  = f'{e}_{template}_all_best_seq'
#     if key_event not in data or key_seq not in data:
#         continue
# 
#     events = data[key_event].squeeze()
#     seqs   = data[key_seq][0]
#     pvals  = data[key_pval].squeeze()
#     pvals_r= data[key_pval_r].squeeze()
#     Goodfit  = data[key_Goodfit].squeeze()
#     slope  = data[key_slope].squeeze()
#     best_seq   = data[key_best_seq].squeeze()
#     if events.ndim == 1:
#         events = np.expand_dims(events, axis=0)
# 
#     for i, event in enumerate(events):
#         if i >= len(seqs):
#             continue
# 
#         pval = np.squeeze(pvals[i])   # [left_pval, right_pval]
#         s = np.squeeze(seqs[i])       # decoded sequence, shape (T,)
#         all_bestseq = np.squeeze(best_seq[i])       # decoded sequence, shape (T,)
# 
#         s_half = s.shape[1] // 2
#         event_time = np.mean(event)  # use midpoint time
# 
#         # Left significant
#         if pval[0] < 0.05 and pvals_r[i][0] < 0.05:
#             left_replay.append(np.argmax(s[:,:s_half], axis=0) )
#             left_times.append(event_time)
#             left_intervals.append(event)
#             left_replay_goodfit.append(Goodfit[i,0])
#             left_replay_slope.append(slope[i][0, 0])
#             left_all_bestseq.append(all_bestseq[:s_half])
#             left_replay_matrix.append(s[:, :s_half])
#         # Right significant
#         if pval[1] < 0.05 and pvals_r[i][1] <0.05:
#             right_replay.append(np.argmax(s[:,s_half:], axis=0))
#             right_times.append(event_time)
#             right_intervals.append(event)
#             right_replay_goodfit.append(Goodfit[i,1])
#             right_replay_slope.append(slope[i][0, 1])
#             right_all_bestseq.append(all_bestseq[s_half:])
#             right_replay_matrix.append(s[:, s_half:]) 
#         if (pval[0] < 0.05 and pvals_r[i][0] < 0.05) or (pval[1] < 0.05 and pvals_r[i][1] <0.05):
#             all_intervals.append(event)

replay ripple non-replay ripple

In [ ]:

combined_intervals = np.array(all_intervals)

In [ ]:
# Compute replay probability in 30-second windows
# For each center_s in ca1_complete_df, compute the probability that a ripple contains a replay

# Get the window size from the D2 parameters (should be 30s)
window_size = 30  # seconds

# Initialize columns for ripple count, replay count, and replay probability
ca1_complete_df['ripple_count'] = 0
ca1_complete_df['replay_count'] = 0
ca1_complete_df['replay_prob'] = np.nan

# Combine all replay times (left + right)
all_replay_times = np.concatenate([
    np.array(left_times),
    np.array(right_times)
])


# For each center time in ca1_complete_df
for idx, row in ca1_complete_df.iterrows():
    center = row['center_s']
    
    # Define window boundaries (center ± window_size/2)
    win_start = center - window_size / 2
    win_end = center + window_size / 2
    
    # Count ripples within this window that fall in NREM
    ripples_in_window = np.sum((ripple_peaks >= win_start) & (ripple_peaks <= win_end))
    
    # Count replays within this window
    replays_in_window = np.sum((all_replay_times >= win_start) & (all_replay_times <= win_end))
    
    # Update the dataframe
    ca1_complete_df.at[idx, 'ripple_count'] = ripples_in_window
    ca1_complete_df.at[idx, 'replay_count'] = replays_in_window
    
    # Compute replay probability (avoid division by zero)
    if ripples_in_window > 0:
        ca1_complete_df.at[idx, 'replay_prob'] = replays_in_window / ripples_in_window
    else:
        ca1_complete_df.at[idx, 'replay_prob'] = np.nan

In [ ]:
# Filter NREM data and find epochs > 3 minutes
nrem_df = ca1_complete_df[ca1_complete_df['state'] == 'NREM'].copy()

# Find continuous NREM epochs by looking at gaps in center_s
nrem_df['time_gap'] = nrem_df['center_s'].diff()
nrem_df['new_epoch'] = (nrem_df['time_gap'] > 60).cumsum()  # Mark new epoch if gap > 60s

# Calculate duration of each epoch
epoch_durations = nrem_df.groupby('new_epoch')['center_s'].agg(['min', 'max'])
epoch_durations['duration'] = epoch_durations['max'] - epoch_durations['min']

# Filter epochs with duration > 3 minutes (180 seconds)
valid_epochs = epoch_durations[epoch_durations['duration'] > 180]

for idx, row in valid_epochs.iterrows():
    pass

# Randomly select one epoch
if len(valid_epochs) > 0:
    selected_epoch_idx = np.random.choice(valid_epochs.index)
    selected_epoch = valid_epochs.loc[selected_epoch_idx]
    
    
    # Select a random 3-minute (180s) window within this epoch
    epoch_start = selected_epoch['min']
    epoch_end = selected_epoch['max']
    max_window_start = epoch_end - 180
    
    if max_window_start > epoch_start:
        window_start = np.random.uniform(epoch_start, max_window_start)
    else:
        window_start = epoch_start
    
    window_end = window_start + 180
    
    
    # Extract data for this window
    window_df = ca1_complete_df[
        (ca1_complete_df['center_s'] >= window_start) & 
        (ca1_complete_df['center_s'] <= window_end)
    ].copy()
    
else:
    pass

In [ ]:
# Filter NREM data and find epochs > 3 minutes
nrem_df = ca1_complete_df[ca1_complete_df['state'] == 'NREM'].copy()

# Find continuous NREM epochs by looking at gaps in center_s
nrem_df['time_gap'] = nrem_df['center_s'].diff()
nrem_df['new_epoch'] = (nrem_df['time_gap'] > 60).cumsum()  # Mark new epoch if gap > 60s

# Calculate duration of each epoch
epoch_durations = nrem_df.groupby('new_epoch')['center_s'].agg(['min', 'max'])
epoch_durations['duration'] = epoch_durations['max'] - epoch_durations['min']

# Filter epochs with duration > 3 minutes (180 seconds)
valid_epochs = epoch_durations[epoch_durations['duration'] > 180]

for idx, row in valid_epochs.iterrows():
    pass

In [ ]:
import numpy as np

# --- parameters ---
window_size = 30.0  # s
delta = 0.08        # s, e.g. ±80 ms around ripple peak

all_replay_times = np.sort(np.concatenate([np.asarray(left_times), np.asarray(right_times)]))
ripple_peaks = np.asarray(ripple_peaks)

def has_replay_near_ripple(rp, replay_times, delta):
    # replay_times sorted
    lo, hi = rp - delta, rp + delta
    i0 = np.searchsorted(replay_times, lo, side="left")
    i1 = np.searchsorted(replay_times, hi, side="right")
    return (i1 > i0)

# Precompute per-ripple labels once
ripple_has_replay = np.array([has_replay_near_ripple(rp, all_replay_times, delta)
                              for rp in ripple_peaks], dtype=bool)

# Initialize
ca1_complete_df["ripple_count"] = 0
ca1_complete_df["replay_ripple_count"] = 0
ca1_complete_df["replay_prob"] = np.nan
ca1_complete_df["ripple_rate"] = np.nan

for idx, center in ca1_complete_df["center_s"].items():
    win_start = center - window_size/2
    win_end   = center + window_size/2

    mask = (ripple_peaks >= win_start) & (ripple_peaks <= win_end)
    n_rip = int(mask.sum())
    n_rep_rip = int(ripple_has_replay[mask].sum())

    ca1_complete_df.at[idx, "ripple_count"] = n_rip
    ca1_complete_df.at[idx, "replay_ripple_count"] = n_rep_rip
    ca1_complete_df.at[idx, "ripple_rate"] = n_rip / window_size
    ca1_complete_df.at[idx, "replay_prob"] = (n_rep_rip / n_rip) if n_rip > 0 else np.nan

In [ ]:
# Plot D2 and Replay Probability vs Time for the 3-minute window
# Randomly select one epoch
if len(valid_epochs) > 0:
    selected_epoch_idx = 16
    selected_epoch = valid_epochs.loc[selected_epoch_idx]
    
    Timelength=60*3
    # Select a random 3-minute (180s) window within this epoch
    epoch_start = 14059
    epoch_end = 14569
    max_window_start = epoch_end - Timelength
    
    if max_window_start > epoch_start:
        window_start = np.random.uniform(epoch_start, max_window_start)
    else:
        window_start = epoch_start
    window_start=14344.6+6
    window_end = window_start + Timelength
    
    
    # Extract data for this window
    window_df = ca1_complete_df[
        (ca1_complete_df['center_s'] >= window_start) & 
        (ca1_complete_df['center_s'] <= window_end)
    ].copy()
    
else:
    pass

In [ ]:
# Z-score within this window

import matplotlib as mpl
# Global font: prefer Arial, with safe fallbacks
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]

# Mathtext to match
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'   # or 'Helvetica'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
window_df["d2_z"]  = (window_df["d2"]  - window_df["d2"].mean())  / window_df["d2"].std()
window_df["MUA_z"] = (window_df["MUA"] - window_df["MUA"].mean()) / window_df["MUA"].std()
# Convert to relative seconds instead of minutes
window_df["time_seconds"] = window_df["center_s"] - window_start

window_df["time_minutes"] = (window_df["center_s"] - window_start) / 60

# Color palette
c_d2 = "#4C72B0"
c_mua = "#8EC9E3"
c_rep = "#691C1B"

fig, ax1 = plt.subplots(figsize=(12, 5))

# --- Left axis: z(D2) and z(MUA) ---
l1, = ax1.plot(
    window_df["time_seconds"], window_df["d2_z"],
    linewidth=2, color=c_d2, alpha=1, label="D2 (z)"
)

l2, = ax1.plot(
    window_df["time_seconds"], window_df["MUA_z"],
    linewidth=2, color=c_mua, alpha=1, label="MUA (z)"
)

ax1.set_ylabel("d2 distance or MUA (z-score)")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.grid(False)

# --- Right axis: Replay probability ---
ax2 = ax1.twinx()
l3, = ax2.plot(
    window_df["time_seconds"], window_df["replay_prob"],
    linewidth=2, color=c_rep, alpha=1, linestyle="--", label="Replay probability"
)

ax2.set_ylabel("Replay probability")
ax2.spines["top"].set_visible(False)
ax2.grid(False)

ax1.set_xlabel("Time (minutes)")
# Force x-axis ticks at 0, 30, 60, 90, 120, 150, 180 s
ax1.set_xticks([0, 30, 60, 90, 120, 150, 180])
ax1.set_xlim(0, 180)
# --- Combined legend ---
ax1.legend([l1, l2, l3],
           ["D2 (z)", "MUA (z)", "Replay probability"],
           loc="upper right")

plt.tight_layout()
savepath = rf'../figure_output\fig2_C_{animal_name}_{day}_D2_MUA_replay_probability_{selected_epoch_idx}.svg'
plt.savefig(savepath, format='svg', dpi=300)
plt.show()